### Mount Drive, Extracting Dataset and imports

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import zipfile, os

ZIP_PATH    = "/content/drive/MyDrive/cig_ps.zip"
EXTRACT_DIR = "/content/cig_ps"

if not os.path.exists(EXTRACT_DIR):
    with zipfile.ZipFile(ZIP_PATH, "r") as z:
        z.extractall(EXTRACT_DIR)
    print("Extracted!")
else:
    print("Already extracted.")

BASE       = "/content/cig_ps/cig_ps"
TRAIN_DIR  = f"{BASE}/train_images"
TEST_DIR   = f"{BASE}/test_images"
LABELS_CSV = f"{BASE}/train-labels.csv"
OUTPUT_DIR = "/content/outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)
print("Paths set.")


In [2]:
import os, random, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from collections import Counter

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

# Reproducibility
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if DEVICE.type == 'cuda':
    torch.cuda.manual_seed(SEED)


### 2.1 Vocabulary & Label Encoding

In [4]:
VOCAB = ['2','3','4','5','6','7','8','9',
             'A','B','C','D','E','F','G','H','J','K',
             'M','N','P','Q','R','S','T','U','V','W','X','Y','Z']
VOCAB_SIZE = len(VOCAB)
SEQ_LEN = 6
CHAR2IDX = {c: i for i, c in enumerate(VOCAB)}
IDX2CHAR = {i: c for c, i in CHAR2IDX.items()}

def encode_label(text):
    return torch.tensor([CHAR2IDX[c] for c in text], dtype=torch.long)

def decode_label(tensor):
    return ''.join(IDX2CHAR[i.item()] for i in tensor)


### 2.2 Load & Split CSV

In [5]:
ANOMALOUS = {'train-2184.png', 'train-6819.png'}

df = pd.read_csv(LABELS_CSV, index_col=0)
df_clean = df[~df['image'].isin(ANOMALOUS)].copy()
df_clean['text'] = df_clean['text'].str.strip()
df_clean = df_clean.reset_index(drop=True)

print(f"Clean samples: {len(df_clean)}")

# 90 / 10 split
df_clean = df_clean.sample(frac=1, random_state=SEED).reset_index(drop=True)
split = int(0.9 * len(df_clean))
df_train = df_clean.iloc[:split].reset_index(drop=True)
df_val = df_clean.iloc[split:].reset_index(drop=True)

print(f"Train : {len(df_train)} Val : {len(df_val)}")


### 2.3 PyTorch Dataset

In [6]:
class CaptchaDataset(Dataset):
    """
    Returns (image_tensor, label_tensor)
    image_tensor : Float32, shape (1, 100, 200), values in [0, 1]
    label_tensor : Long,    shape (6,)
    """
    def __init__(self, df, img_dir, augment=False):
        self.df = df.reset_index(drop=True)
        self.img_dir = img_dir
        self.augment = augment

        self.base_tf = T.Compose([
            T.Grayscale(),
            T.ToTensor(),          # (1, H, W), [0,1]
        ])
        self.aug_tf = T.Compose([
            T.Grayscale(),
            T.RandomAffine(degrees=3, translate=(0.02, 0.02), scale=(0.97, 1.03)),
            T.ToTensor(),
        ])

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(os.path.join(self.img_dir, row['image'])).convert('RGB')
        tf = self.aug_tf if self.augment else self.base_tf
        img_t = tf(img)                         # (1, 100, 200)
        lbl_t = encode_label(row['text'])        # (6,)
        return img_t, lbl_t


class CaptchaTestDataset(Dataset):
    """Test set — no labels."""
    def __init__(self, img_dir):
        self.img_dir = img_dir
        self.files = sorted(os.listdir(img_dir))
        self.tf = T.Compose([T.Grayscale(), T.ToTensor()])

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        fname = self.files[idx]
        img = Image.open(os.path.join(self.img_dir, fname)).convert('RGB')
        return self.tf(img), fname


# ── Instantiate
BATCH_SIZE = 64

train_ds = CaptchaDataset(df_train, TRAIN_DIR, augment=True)
val_ds = CaptchaDataset(df_val,   TRAIN_DIR, augment=False)
test_ds = CaptchaTestDataset(TEST_DIR)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=2, pin_memory=True)
val_loader = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=2, pin_memory=True)
test_loader = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=2, pin_memory=True)

print(f"Train batches : {len(train_loader)}")
print(f"Val batches : {len(val_loader)}")
print(f"Test batches : {len(test_loader)}")

# Verify one batch shape
imgs, lbls = next(iter(train_loader))
print(f"\nBatch image shape : {imgs.shape}   (B, C, H, W)")
print(f"Batch label shape : {lbls.shape}   (B, SEQ_LEN)")
print(f"Pixel range : [{imgs.min():.3f}, {imgs.max():.3f}]")


### 2.4 CER Metric

In [9]:
def levenshtein(s1, s2):
    m, n = len(s1), len(s2)
    dp = list(range(n + 1))
    for i in range(1, m + 1):
        prev, dp[0] = dp[0], i
        for j in range(1, n + 1):
            temp = dp[j]
            dp[j] = prev if s1[i-1] == s2[j-1] else 1 + min(prev, dp[j], dp[j-1])
            prev = temp
    return dp[n]

def compute_cer(preds, targets):
    total_dist, total_len = 0, 0
    for p, t in zip(preds, targets):
        total_dist += levenshtein(p, t)
        total_len  += len(t)
    return total_dist / total_len if total_len > 0 else 0.0

## Baseline Model
### 6-Head CNN

In [ ]:
class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch, pool=True):
        super().__init__()
        layers = [
            nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        ]
        if pool:
            layers.append(nn.MaxPool2d(2, 2))
        self.block = nn.Sequential(*layers)

    def forward(self, x):
        return self.block(x)


class SixHeadCNN(nn.Module):
    def __init__(self, vocab_size=31, seq_len=6):
        super().__init__()
        self.seq_len    = seq_len
        self.vocab_size = vocab_size

        # Feature extractor
        self.features = nn.Sequential(
            ConvBlock(1,   32),
            ConvBlock(32,  64),
            ConvBlock(64, 128),
            ConvBlock(128,256),
        )

        self.pool = nn.AdaptiveAvgPool2d((2, 4))

        feat_dim = 256 * 2 * 4

        self.shared_fc = nn.Sequential(
            nn.Flatten(),
            nn.Linear(feat_dim, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.4),
        )

        # 6 independent heads
        self.heads = nn.ModuleList([
            nn.Linear(512, vocab_size) for _ in range(seq_len)
        ])

    def forward(self, x):
        x = self.features(x)
        x = self.pool(x)
        x = self.shared_fc(x)
        return [head(x) for head in self.heads]


# check
model = SixHeadCNN(VOCAB_SIZE, SEQ_LEN).to(DEVICE)
dummy = torch.zeros(4, 1, 100, 200).to(DEVICE)
outs  = model(dummy)
print(f"Output: {len(outs)} heads, each shape {outs[0].shape}")

total_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {total_params:,}")


### Loss, Optimizer, LR Scheduler

In [12]:
criterion = nn.CrossEntropyLoss()

optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)

# Cosine annealing over training epochs
NUM_EPOCHS = 30
scheduler  = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS, eta_min=1e-5)

### 3.3 Train & Evaluate Functions

In [13]:
def train_one_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss = 0.0
    for imgs, lbls in loader:
        imgs = imgs.to(DEVICE)
        lbls = lbls.to(DEVICE)          # (B, 6)

        optimizer.zero_grad()
        logits = model(imgs)            # list of 6 x (B, 31)

        # Sum cross-entropy over 6 positions
        loss = sum(criterion(logits[i], lbls[:, i]) for i in range(SEQ_LEN))
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(loader)


def evaluate(model, loader):
    model.eval()
    total_loss = 0.0
    all_preds, all_targets = [], []

    with torch.no_grad():
        for imgs, lbls in loader:
            imgs = imgs.to(DEVICE)
            lbls = lbls.to(DEVICE)

            logits = model(imgs)
            loss   = sum(criterion(logits[i], lbls[:, i]) for i in range(SEQ_LEN))
            total_loss += loss.item()

            # Decode predictions
            preds = torch.stack([l.argmax(dim=1) for l in logits], dim=1)  # (B, 6)
            for pred, target in zip(preds, lbls):
                all_preds.append(decode_label(pred))
                all_targets.append(decode_label(target))

    cer = compute_cer(all_preds, all_targets)
    return total_loss / len(loader), cer


### 3.4 Training Loop

In [ ]:
history = {'train_loss': [], 'val_loss': [], 'val_cer': []}
best_cer   = float('inf')
CKPT_PATH  = f"{OUTPUT_DIR}/baseline_best.pth"

print(f"{'Epoch':>5} | {'Train Loss':>10} | {'Val Loss':>9} | {'Val CER':>8} | {'LR':>8} | {'Time':>6}")
print("-" * 65)

for epoch in range(1, NUM_EPOCHS + 1):
    t0 = time.time()

    train_loss = train_one_epoch(model, train_loader, optimizer, criterion)
    val_loss, val_cer = evaluate(model, val_loader)
    scheduler.step()

    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['val_cer'].append(val_cer)

    lr = scheduler.get_last_lr()[0]
    elapsed = time.time() - t0

    print(f"{epoch:>5} | {train_loss:>10.4f} | {val_loss:>9.4f} | {val_cer:>8.4f} | {lr:>8.6f} | {elapsed:>5.1f}s")

    if val_cer < best_cer:
        best_cer = val_cer
        torch.save(model.state_dict(), CKPT_PATH)
        print(f"        ✓ Saved best model  (CER={best_cer:.4f})")

print(f"\nBest Val CER: {best_cer:.4f}")

### 3.5 Training Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].plot(history['train_loss'], label='Train Loss')
axes[0].plot(history['val_loss'],   label='Val Loss')
axes[0].set_title('Loss'); axes[0].set_xlabel('Epoch')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot(history['val_cer'], color='darkorange', label='Val CER')
axes[1].set_title('Validation CER'); axes[1].set_xlabel('Epoch')
axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/baseline_curves.png", dpi=110)
plt.show()
print(f"Best Val CER: {best_cer:.4f}")
